--- Day 10: Hoof It ---
You all arrive at a Lava Production Facility on a floating island in the sky. As the others begin to search the massive industrial complex, you feel a small nose boop your leg and look down to discover a reindeer wearing a hard hat.

The reindeer is holding a book titled "Lava Island Hiking Guide". However, when you open the book, you discover that most of it seems to have been scorched by lava! As you're about to ask how you can help, the reindeer brings you a blank topographic map of the surrounding area (your puzzle input) and looks up at you excitedly.

Perhaps you can help fill in the missing hiking trails?

The topographic map indicates the height at each position using a scale from 0 (lowest) to 9 (highest). For example:

0123
1234
8765
9876
Based on un-scorched scraps of the book, you determine that a good hiking trail is as long as possible and has an even, gradual, uphill slope. For all practical purposes, this means that a hiking trail is any path that starts at height 0, ends at height 9, and always increases by a height of exactly 1 at each step. Hiking trails never include diagonal steps - only up, down, left, or right (from the perspective of the map).

You look up from the map and notice that the reindeer has helpfully begun to construct a small pile of pencils, markers, rulers, compasses, stickers, and other equipment you might need to update the map with hiking trails.

A trailhead is any position that starts one or more hiking trails - here, these positions will always have height 0. Assembling more fragments of pages, you establish that a trailhead's score is the number of 9-height positions reachable from that trailhead via a hiking trail. In the above example, the single trailhead in the top left corner has a score of 1 because it can reach a single 9 (the one in the bottom left).

This trailhead has a score of 2:

...0...
...1...
...2...
6543456
7.....7
8.....8
9.....9
(The positions marked . are impassable tiles to simplify these examples; they do not appear on your actual topographic map.)

This trailhead has a score of 4 because every 9 is reachable via a hiking trail except the one immediately to the left of the trailhead:

..90..9
...1.98
...2..7
6543456
765.987
876....
987....
This topographic map contains two trailheads; the trailhead at the top has a score of 1, while the trailhead at the bottom has a score of 2:

10..9..
2...8..
3...7..
4567654
...8..3
...9..2
.....01
Here's a larger example:

89010123
78121874
87430965
96549874
45678903
32019012
01329801
10456732
This larger example has 9 trailheads. Considering the trailheads in reading order, they have scores of 5, 6, 5, 3, 1, 3, 5, 3, and 5. Adding these scores together, the sum of the scores of all trailheads is 36.

The reindeer gleefully carries over a protractor and adds it to the pile. What is the sum of the scores of all trailheads on your topographic map?

In [1]:
# I believe this problem can be easiest handled by going from 9 to 0: All 9's can reach 1 9, so they get a score of 9. All 8's get the score of the number of nines they reach. All seven get the sum of the scores of the 8's they can reach and so on. It's like path counting from that other exercise. I could prob also count from 0 to 9 and get the same total number of trails, but the puzzle description is interested in trailheads so might as well count them.

In [2]:
with open('input10.txt') as file:
    data = [line.rstrip() for line in file]
data

['5654348767654328943210568976534369910789674323105678',
 '8701239458901217658121477687623478823658789410234089',
 '9687012344567806789054389598510562734569876501870123',
 '7896543569478910690343201423421051650121965432965434',
 '3234512478321001541252102310436710543230892396566985',
 '0105601309010192432567010121045827652340181087654876',
 '3238711210520983401478121012310938981653276321049965',
 '4589100345601276512389652983429845670787345438038734',
 '7671018986798345601456743876536762103894421589125621',
 '8562327019887016512567895805445653214545430677894100',
 '9443456523456325434898656914560344367656528912763210',
 '2356932108701236721019847823271255698767817603454323',
 '1047823019854349832478739854180766789678906541065410',
 '2154014323769858210567026765099850176543215432878929',
 '3763005458958765523450110623458901289012103898945678',
 '7892176767567894654543210210567110378650112367630567',
 '4567989853018723743210321671003025496743203456721656',
 '32987899143296108949874345823

In [3]:
grid_size = len(data)
len(data), len(data[0])

(52, 52)

In [4]:
class Position:

    def __init__(self, height, coordinates, ledger):
        self.height = int(height)
        self.pos = coordinates
        self.x = self.pos[0]
        self.y = self.pos[1]
        self.ledger = ledger # shared across instances with same ledger dict

    @staticmethod
    def build_nodes(grid, ledger):
        for i, line in enumerate(data):
            for j, height in enumerate(line):
                ledger[(i, j)] = Position(height, (i, j), ledger)
        

    def find_neighbors(self):
        neighbor_coords = [ (self.x + 1, self.y),
                            (self.x - 1, self.y),
                            (self.x, self.y + 1),
                            (self.x, self.y - 1)]
        
        self.neighbors = set()
        for ne in neighbor_coords:
            if ne in self.ledger:
                self.neighbors.add(self.ledger[ne])

    def count_connections(self):
        self.reach_peaks = set()
        if self.height == 9:
            self.reach_peaks.add(self)
            self.paths = 1
            return
        else:
            self.paths = 0
            for neighbor in self.neighbors:
                if neighbor.height == self.height + 1:
                    self.paths += neighbor.paths
                    self.reach_peaks.update(neighbor.reach_peaks)
        

In [5]:
ledger = {}
Position.build_nodes(data, ledger)
for node in ledger.values():
    node.find_neighbors()
ledger

{(0, 0): <__main__.Position at 0x108caa7b0>,
 (0, 1): <__main__.Position at 0x10819b890>,
 (0, 2): <__main__.Position at 0x108c72e90>,
 (0, 3): <__main__.Position at 0x10896f820>,
 (0, 4): <__main__.Position at 0x108421cd0>,
 (0, 5): <__main__.Position at 0x108244290>,
 (0, 6): <__main__.Position at 0x10818f570>,
 (0, 7): <__main__.Position at 0x10818f790>,
 (0, 8): <__main__.Position at 0x108c5bb50>,
 (0, 9): <__main__.Position at 0x108c5bc50>,
 (0, 10): <__main__.Position at 0x108b066c0>,
 (0, 11): <__main__.Position at 0x10818b110>,
 (0, 12): <__main__.Position at 0x108ed1b70>,
 (0, 13): <__main__.Position at 0x108d5dd30>,
 (0, 14): <__main__.Position at 0x108cacef0>,
 (0, 15): <__main__.Position at 0x108b39c10>,
 (0, 16): <__main__.Position at 0x108b39fd0>,
 (0, 17): <__main__.Position at 0x108dc97b0>,
 (0, 18): <__main__.Position at 0x108dc9910>,
 (0, 19): <__main__.Position at 0x11104fd90>,
 (0, 20): <__main__.Position at 0x11104fed0>,
 (0, 21): <__main__.Position at 0x108f5ab10>

In [6]:
num_paths = 0
for elev in range(9, -1, -1):
    for node in ledger.values():
        if node.height == elev:
            node.count_connections()
            if elev == 0:
                num_paths += node.paths
num_paths

1494

In [7]:
# Ok this was elegant but not what was asked for: We count paths even if they have the same start and end. Let's make sure we count the number of start - end pairs instead. I adapted the count_connections function above a bit so here we go:

sum_peaks = 0
for elev in range(9, -1, -1):
    for node in ledger.values():
        if node.height == elev:
            node.count_connections()
            if elev == 0:
                sum_peaks += len(node.reach_peaks)
sum_peaks

646

Your puzzle answer was 646.

The first half of this puzzle is complete! It provides one gold star: *

--- Part Two ---
The reindeer spends a few minutes reviewing your hiking trail map before realizing something, disappearing for a few minutes, and finally returning with yet another slightly-charred piece of paper.

The paper describes a second way to measure a trailhead called its rating. A trailhead's rating is the number of distinct hiking trails which begin at that trailhead. For example:

.....0.
..4321.
..5..2.
..6543.
..7..4.
..8765.
..9....
The above map has a single trailhead; its rating is 3 because there are exactly three distinct hiking trails which begin at that position:

.....0.   .....0.   .....0.
..4321.   .....1.   .....1.
..5....   .....2.   .....2.
..6....   ..6543.   .....3.
..7....   ..7....   .....4.
..8....   ..8....   ..8765.
..9....   ..9....   ..9....
Here is a map containing a single trailhead with rating 13:

..90..9
...1.98
...2..7
6543456
765.987
876....
987....
This map contains a single trailhead with rating 227 (because there are 121 distinct hiking trails that lead to the 9 on the right edge and 106 that lead to the 9 on the bottom edge):

012345
123456
234567
345678
4.6789
56789.
Here's the larger example from before:

89010123
78121874
87430965
96549874
45678903
32019012
01329801
10456732
Considering its trailheads in reading order, they have ratings of 20, 24, 10, 4, 1, 4, 5, 8, and 5. The sum of all trailhead ratings in this larger example topographic map is 81.

You're not sure how, but the reindeer seems to have crafted some tiny flags out of toothpicks and bits of paper and is using them to mark trailheads on your topographic map. What is the sum of the ratings of all trailheads?



In [8]:
# Well we have inadvertantly answered that question before.
num_paths

1494

That's the right answer! You are one gold star closer to finding the Chief Historian.

You have completed Day 10! You can [Share] this victory or [Return to Your Advent Calendar].